合成光谱数据生成器
本笔记本用于生成蛋白质二级结构分析的合成中红外光谱数据。

功能特点
- **α-螺旋**: 1个洛伦兹峰，中心在1656±2 cm⁻¹
- **β-折叠**: 选择2个标准中心，每个中心生成3-5个子峰
- **β-转角**: 选择1-2个标准中心，每个中心生成3-5个子峰
- **其他峰**: 在非样本区域生成4-7个额外峰
- **复合光谱**: 所有组分的和加上真实噪声和基线漂移

物理建模
- 高斯+多普勒展宽
- 多层次噪声模型
- 复杂的基线漂移
- 微扰动模拟


1. 导入依赖库

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Sequence, Tuple
import os
import yaml

import numpy as np
from scipy import integrate
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import seaborn as sns


2. 加载配置文件


In [2]:
# 加载配置文件
def load_config(config_path: str = "generate_dataset.yml"):
    """加载YAML配置文件"""
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    return config

# 加载配置
config = load_config()

# 从配置中提取参数
WAVENUM_MIN = config['wavenumber']['min']
WAVENUM_MAX = config['wavenumber']['max']
STEP = config['wavenumber']['step']

SUBPEAKS_RANGE = tuple(config['envelope']['subpeaks_range'])
AMP_RANGE = tuple(config['envelope']['amplitude_range'])
LORENTZ_FWHM_RANGE = tuple(config['envelope']['lorentz_fwhm_range'])

GAUSS_FWHM_RANGE = tuple(config['broadening']['gauss_fwhm_range'])
DOPPLER_FWHM_RANGE = tuple(config['broadening']['doppler_fwhm_range'])

NOISE_STD_REL = config['noise']['detector_noise_std']
MICRO_PERTURBATION_STD = config['noise']['micro_perturbation_std']

BASELINE_DRIFT_AMP = config['baseline_drift']['amplitude']
BASELINE_DRIFT_PERIOD = config['baseline_drift']['period']
RANDOM_DRIFT_STD = config['baseline_drift']['random_drift_std']

# 标准峰值定义
ALPHA_HELIX_PEAK = (config['peaks']['alpha_helix']['center'], 
                    config['peaks']['alpha_helix']['tolerance'])
BETA_SHEET_PEAKS = [tuple(peak) for peak in config['peaks']['beta_sheet']]
BETA_TURN_PEAKS = [tuple(peak) for peak in config['peaks']['beta_turn']]

# 其他峰参数
OTHER_PEAKS_REGIONS = [tuple(region) for region in config['other_peaks']['regions']]
OTHER_PEAKS_COUNT_RANGE = tuple(config['other_peaks']['count_range'])
OTHER_PEAKS_AMP_RATIO = tuple(config['other_peaks']['amplitude_ratio'])

# 数据集参数
N_SAMPLES = config['dataset']['n_samples']
OUTPUT_FILENAME = config['dataset']['output_filename']
RANDOM_SEED = config['dataset']['random_seed']

# 设置绘图样式
plt.style.use(config['visualization']['style'])
sns.set_palette(config['visualization']['palette'])

print("配置文件加载完成！")
print(f"波数范围: {WAVENUM_MIN} - {WAVENUM_MAX} cm⁻¹")
print(f"波数步长: {STEP} cm⁻¹")
print(f"子峰范围: {SUBPEAKS_RANGE[0]}-{SUBPEAKS_RANGE[1]} 个/标准中心")
print(f"噪声水平: {NOISE_STD_REL*100:.1f}%")
print(f"生成样本数: {N_SAMPLES}")


KeyError: 'other_peaks'

3. 内部工具函数

In [ ]:
# 内部工具常量
_ln2 = np.log(2.0)
FWHM_TO_SIGMA = 1.0 / (2.0 * np.sqrt(2.0 * _ln2))

def _lorentzian(x: np.ndarray, x0: float, gamma: float, amp: float) -> np.ndarray:
    """生成洛伦兹峰。"""
    half_gamma = 0.5 * gamma
    return amp * (half_gamma ** 2) / ((x - x0) ** 2 + half_gamma ** 2)

def _gaussian_kernel(sigma: float) -> np.ndarray:
    """生成用于展宽的高斯核。"""
    half_pts = int(np.ceil(3 * sigma / STEP))
    grid = np.arange(-half_pts, half_pts + 1) * STEP
    k = np.exp(-0.5 * (grid / sigma) ** 2)
    return k / k.sum()

def _blur(signal: np.ndarray, sigma_instr: float, sigma_dopp: float) -> np.ndarray:
    """对信号应用高斯展宽。"""
    sigma_tot = np.sqrt(sigma_instr ** 2 + sigma_dopp ** 2)
    return np.convolve(signal, _gaussian_kernel(sigma_tot), mode="same")


4. 其他峰生成函数

In [ ]:
def _generate_other_peaks(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """在非样本区域生成其他峰。"""
    n_peaks = rng.integers(*OTHER_PEAKS_COUNT_RANGE)
    other_envelope = np.zeros_like(x)
    
    for _ in range(n_peaks):
        # 随机选择一个区域
        region = rng.choice(OTHER_PEAKS_REGIONS)
        centre = rng.uniform(region[0], region[1])
        
        # 生成峰值参数
        fwhm = rng.uniform(*LORENTZ_FWHM_RANGE)
        amp = rng.uniform(*OTHER_PEAKS_AMP_RATIO) * rng.uniform(*AMP_RANGE)
        
        # 添加峰
        other_envelope += _lorentzian(x, centre, fwhm, amp)
    
    # 应用展宽
    g_fwhm = rng.uniform(*GAUSS_FWHM_RANGE)
    d_fwhm = rng.uniform(*DOPPLER_FWHM_RANGE)
    return _blur(other_envelope, g_fwhm * FWHM_TO_SIGMA, d_fwhm * FWHM_TO_SIGMA)

print("其他峰生成函数定义完成！")


5. 基线漂移和噪声生成函数

In [ ]:
def _generate_baseline_drift(x: np.ndarray, rng: np.random.Generator, max_amp: float) -> np.ndarray:
    """生成真实的基线漂移。"""
    # 生成多个频率分量以获得真实的漂移
    drift = np.zeros_like(x)
    
    # 低频漂移 (慢基线变化)
    freq1 = 2 * np.pi / BASELINE_DRIFT_PERIOD
    phase1 = rng.uniform(0, 2 * np.pi)
    amp1 = rng.uniform(0.5, 1.0) * BASELINE_DRIFT_AMP * max_amp
    drift += amp1 * np.sin(freq1 * x + phase1)
    
    # 中频漂移
    freq2 = 2 * np.pi / (BASELINE_DRIFT_PERIOD * 0.5)
    phase2 = rng.uniform(0, 2 * np.pi)
    amp2 = rng.uniform(0.3, 0.7) * BASELINE_DRIFT_AMP * max_amp
    drift += amp2 * np.sin(freq2 * x + phase2)
    
    # 添加一些随机低频噪声
    random_drift = rng.normal(0, RANDOM_DRIFT_STD * max_amp, size=x.size)
    # 平滑随机漂移
    random_drift = gaussian_filter1d(random_drift, sigma=5)
    
    return drift + random_drift

def _add_realistic_noise(signal: np.ndarray, rng: np.random.Generator, max_amp: float) -> np.ndarray:
    """添加真实噪声，包括检测器噪声和微扰动。"""
    # 检测器噪声 (白噪声)
    detector_noise = rng.normal(0, NOISE_STD_REL * max_amp, size=signal.size)
    
    # 峰值位置的微扰动 (模拟采样误差)
    micro_noise = np.zeros_like(signal)
    for i in range(1, signal.size - 1):
        if signal[i] > signal[i-1] and signal[i] > signal[i+1]:  # 峰值位置
            # 使用绝对值确保正尺度参数
            perturbation_std = MICRO_PERTURBATION_STD * abs(signal[i])
            micro_noise[i] = rng.normal(0, perturbation_std)  # 峰值的微扰动
    
    # 组合所有噪声分量
    return signal + detector_noise + micro_noise

print("基线漂移和噪声生成函数定义完成！")


6. 包络构建函数

In [ ]:
def _build_envelope(
    x: np.ndarray,
    peaks,                        # α-螺旋元组 OR β-折叠/β-转角列表
    rng: np.random.Generator,
    n_canonical: int = 1          # 选择多少个标准中心
) -> np.ndarray:
    """返回二级结构的展宽包络。"""

    # -------- 选择标准中心 --------
    if isinstance(peaks[0], tuple):  # β-折叠 / β-转角列表
        canonical_choices = rng.choice(peaks, size=n_canonical, replace=False)
    else:                            # α-螺旋单个元组
        canonical_choices = [peaks]

    # 在标准组之间分配总振幅
    total_amp = rng.uniform(*AMP_RANGE)
    canon_weights = rng.dirichlet(np.ones(len(canonical_choices)))

    envelope = np.zeros_like(x)

    # -------- 对于每个标准中心 --------
    for (centre0, tol), weight in zip(canonical_choices, canon_weights):
        n_sub = rng.integers(SUBPEAKS_RANGE[0], SUBPEAKS_RANGE[1] + 1)
        sub_weights = rng.dirichlet(np.ones(n_sub))  # 每个子峰的分数

        for frac in sub_weights:
            centre = centre0 + rng.uniform(-tol, tol)
            fwhm = rng.uniform(*LORENTZ_FWHM_RANGE)
            amp = total_amp * weight * frac
            envelope += _lorentzian(x, centre, fwhm, amp)

    # -------- 高斯 + 多普勒展宽 --------
    g_fwhm = rng.uniform(*GAUSS_FWHM_RANGE)
    d_fwhm = rng.uniform(*DOPPLER_FWHM_RANGE)
    return _blur(envelope, g_fwhm * FWHM_TO_SIGMA, d_fwhm * FWHM_TO_SIGMA)

def _build_beta_turn_split(
    x: np.ndarray,
    rng: np.random.Generator
) -> Tuple[np.ndarray, np.ndarray]:
    """构建可能分割为lo/hi区域的β-转角。"""
    # 随机选择1或2个峰
    n_peaks = rng.integers(1, 3)  # 1或2个峰
    
    if n_peaks == 1:
        # 单峰 - 随机分配给lo或hi
        if rng.random() < 0.5:
            lo_envelope = _build_envelope(x, BETA_TURN_PEAKS, rng, n_canonical=1)
            hi_envelope = np.zeros_like(x)
        else:
            lo_envelope = np.zeros_like(x)
            hi_envelope = _build_envelope(x, BETA_TURN_PEAKS, rng, n_canonical=1)
    else:
        # 两个峰 - 在lo和hi之间分割
        lo_envelope = _build_envelope(x, BETA_TURN_PEAKS, rng, n_canonical=1)
        hi_envelope = _build_envelope(x, BETA_TURN_PEAKS, rng, n_canonical=1)
    
    return lo_envelope, hi_envelope

print("包络构建函数定义完成！")


7. 数据结构定义

In [ ]:
@dataclass
class DatasetSample:
    """数据集样本数据结构"""
    alpha_helix: np.ndarray
    beta_sheet_lo: np.ndarray
    beta_sheet_hi: np.ndarray
    beta_turn_lo: np.ndarray
    beta_turn_hi: np.ndarray
    other_peaks: np.ndarray
    composite: np.ndarray

print("数据结构定义完成！")


8. 数据集生成器类

In [ ]:
class Dataset2Generator:
    """为dataset2.npz生成合成光谱样本。"""

    def __init__(self, seed: int | None = None):
        self.rng = np.random.default_rng(seed)
        self.x = np.arange(WAVENUM_MIN, WAVENUM_MAX + STEP, STEP)
        print(f"生成器初始化完成！波数轴长度: {len(self.x)}")

    def generate_sample(self) -> DatasetSample:
        """生成单个样本"""
        # 生成α-螺旋 (1个峰)
        alpha = _build_envelope(self.x, ALPHA_HELIX_PEAK, self.rng, n_canonical=1)
        
        # 生成β-折叠 (2个峰，分割为lo/hi)
        beta_s_lo = _build_envelope(self.x, BETA_SHEET_PEAKS, self.rng, n_canonical=1)
        beta_s_hi = _build_envelope(self.x, BETA_SHEET_PEAKS, self.rng, n_canonical=1)
        
        # 生成β-转角 (1-2个峰，分割为lo/hi)
        beta_t_lo, beta_t_hi = _build_beta_turn_split(self.x, self.rng)
        
        # 生成其他峰
        other = _generate_other_peaks(self.x, self.rng)
        
        # 创建清洁复合光谱 (无噪声/漂移)
        composite_clean = alpha + beta_s_lo + beta_s_hi + beta_t_lo + beta_t_hi + other
        
        # 仅对复合光谱添加基线漂移和噪声
        baseline_drift = _generate_baseline_drift(self.x, self.rng, composite_clean.max())
        composite_with_drift = composite_clean + baseline_drift
        
        # 添加真实噪声
        composite_noisy = _add_realistic_noise(composite_with_drift, self.rng, composite_clean.max())

        return DatasetSample(
            alpha_helix=alpha,  # 保持清洁
            beta_sheet_lo=beta_s_lo,  # 保持清洁
            beta_sheet_hi=beta_s_hi,  # 保持清洁
            beta_turn_lo=beta_t_lo,  # 保持清洁
            beta_turn_hi=beta_t_hi,  # 保持清洁
            other_peaks=other,  # 保持清洁
            composite=composite_noisy  # 仅此有噪声/漂移
        )

print("数据集生成器类定义完成！")


9. 批量数据集生成函数

In [ ]:
def generate_dataset(self, n_samples: int = 10000):
    """生成完整数据集"""
    print(f"正在生成 {n_samples} 个样本...")
    
    # 初始化数组
    alpha_all = np.zeros((n_samples, self.x.size))
    beta_s_lo_all = np.zeros((n_samples, self.x.size))
    beta_s_hi_all = np.zeros((n_samples, self.x.size))
    beta_t_lo_all = np.zeros((n_samples, self.x.size))
    beta_t_hi_all = np.zeros((n_samples, self.x.size))
    other_all = np.zeros((n_samples, self.x.size))
    composite_all = np.zeros((n_samples, self.x.size))
    
    for i in range(n_samples):
        if i % 1000 == 0:
            print(f"已生成 {i}/{n_samples} 个样本...")
        
        samp = self.generate_sample()
        alpha_all[i] = samp.alpha_helix
        beta_s_lo_all[i] = samp.beta_sheet_lo
        beta_s_hi_all[i] = samp.beta_sheet_hi
        beta_t_lo_all[i] = samp.beta_turn_lo
        beta_t_hi_all[i] = samp.beta_turn_hi
        other_all[i] = samp.other_peaks
        composite_all[i] = samp.composite
    
    print("数据集生成完成！")
    
    return {
        'wavenumber_axis': self.x,
        'alpha_helix': alpha_all,
        'beta_sheet_lo': beta_s_lo_all,
        'beta_sheet_hi': beta_s_hi_all,
        'beta_turn_lo': beta_t_lo_all,
        'beta_turn_hi': beta_t_hi_all,
        'other_peaks': other_all,
        'composite': composite_all
    }

def save_dataset(self, n_samples: int = 10000, filename: str = 'dataset2.npz'):
    """生成并保存数据集到.npz文件"""
    dataset = self.generate_dataset(n_samples)
    
    # 保存到.npz文件
    save_path = os.path.join(os.path.dirname(__file__), filename)
    np.savez(save_path, **dataset)
    
    print(f"数据集已保存到: {save_path}")
    print(f"数据集形状: {n_samples} 个样本 × {self.x.size} 个光谱点")
    print(f"波数范围: {self.x.min():.1f} 到 {self.x.max():.1f} cm⁻¹")
    
    return save_path

# 将函数绑定到类
Dataset2Generator.generate_dataset = generate_dataset
Dataset2Generator.save_dataset = save_dataset

print("批量数据集生成函数定义完成！")


10. 生成单个样本并可视化

In [ ]:
# 创建生成器实例
generator = Dataset2Generator(seed=RANDOM_SEED)

# 生成单个样本
sample = generator.generate_sample()

# 可视化单个样本
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('合成光谱样本示例', fontsize=16, fontweight='bold')

# 各组分光谱
axes[0, 0].plot(generator.x, sample.alpha_helix, label='α-螺旋', linewidth=2)
axes[0, 0].plot(generator.x, sample.beta_sheet_lo, label='β-折叠 (lo)', linewidth=2)
axes[0, 0].plot(generator.x, sample.beta_sheet_hi, label='β-折叠 (hi)', linewidth=2)
axes[0, 0].plot(generator.x, sample.beta_turn_lo, label='β-转角 (lo)', linewidth=2)
axes[0, 0].plot(generator.x, sample.beta_turn_hi, label='β-转角 (hi)', linewidth=2)
axes[0, 0].plot(generator.x, sample.other_peaks, label='其他峰', linewidth=2)
axes[0, 0].set_xlabel('波数 (cm⁻¹)')
axes[0, 0].set_ylabel('强度')
axes[0, 0].set_title('各组分光谱')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 清洁复合光谱
composite_clean = (sample.alpha_helix + sample.beta_sheet_lo + sample.beta_sheet_hi + 
                   sample.beta_turn_lo + sample.beta_turn_hi + sample.other_peaks)
axes[0, 1].plot(generator.x, composite_clean, label='清洁复合光谱', linewidth=2, color='blue')
axes[0, 1].set_xlabel('波数 (cm⁻¹)')
axes[0, 1].set_ylabel('强度')
axes[0, 1].set_title('清洁复合光谱')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 带噪声的复合光谱
axes[1, 0].plot(generator.x, sample.composite, label='带噪声复合光谱', linewidth=2, color='red')
axes[1, 0].plot(generator.x, composite_clean, label='清洁复合光谱', linewidth=1, color='blue', alpha=0.7)
axes[1, 0].set_xlabel('波数 (cm⁻¹)')
axes[1, 0].set_ylabel('强度')
axes[1, 0].set_title('带噪声 vs 清洁复合光谱')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 噪声分析
noise = sample.composite - composite_clean
axes[1, 1].plot(generator.x, noise, label='噪声', linewidth=1, color='gray')
axes[1, 1].set_xlabel('波数 (cm⁻¹)')
axes[1, 1].set_ylabel('噪声强度')
axes[1, 1].set_title(f'噪声分析 (RMS: {np.sqrt(np.mean(noise**2)):.4f})')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"样本生成完成！")
print(f"各组分强度: α={np.max(sample.alpha_helix):.3f}, β_lo={np.max(sample.beta_sheet_lo):.3f}, β_hi={np.max(sample.beta_sheet_hi):.3f}")
print(f"复合光谱强度: 清洁={np.max(composite_clean):.3f}, 带噪声={np.max(sample.composite):.3f}")


11. 生成小规模测试数据集

In [ ]:
# 生成小规模测试数据集
print("生成小规模测试数据集...")
test_dataset = generator.generate_dataset(n_samples=100)

# 可视化数据集统计信息
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('测试数据集统计信息', fontsize=16, fontweight='bold')

# 各组分强度分布
components = ['alpha_helix', 'beta_sheet_lo', 'beta_sheet_hi', 'beta_turn_lo', 'beta_turn_hi', 'other_peaks']
component_names = ['α-螺旋', 'β-折叠(lo)', 'β-折叠(hi)', 'β-转角(lo)', 'β-转角(hi)', '其他峰']

for i, (comp, name) in enumerate(zip(components, component_names)):
    row, col = i // 3, i % 3
    max_intensities = np.max(test_dataset[comp], axis=1)
    axes[row, col].hist(max_intensities, bins=20, alpha=0.7, edgecolor='black')
    axes[row, col].set_xlabel('最大强度')
    axes[row, col].set_ylabel('频次')
    axes[row, col].set_title(f'{name} 强度分布')
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"测试数据集生成完成！")
print(f"数据集形状: {test_dataset['composite'].shape}")
print(f"各组分平均强度:")
for comp, name in zip(components, component_names):
    avg_intensity = np.mean(np.max(test_dataset[comp], axis=1))
    print(f"  {name}: {avg_intensity:.3f}")


12. 生成完整数据集

In [ ]:
# 生成完整数据集
print("开始生成完整数据集...")
save_path = generator.save_dataset(n_samples=N_SAMPLES, filename=OUTPUT_FILENAME)

print(f"\nDataset2.npz 已成功生成并保存！")
print(f"文件位置: {save_path}")
print(f"文件大小: {os.path.getsize(save_path) / (1024*1024):.1f} MB")
